# Carcinogen Harmonizer Runner

This notebook runs the carcinogens enrichment pipeline end-to-end.
It starts from a curated seed CSV that anchors stable IDs, labels, and provenance, then optionally expands from IARC and other sources.

It supports:
- seed normalization
- optional IARC expansion
- optional PubChem enrichment (CID, SMILES, InChI/InChIKey, CAS, properties)
- optional ClassyFire / ChemOnt enrichment (validated chemical class with references)
- optional local reference enrichment (NTP/OEHHA/CTD)
- QA + graph outputs
- CSV and/or JSON exports

In [1]:
from pathlib import Path
import json
import pandas as pd

from carcinogen_harmonizer import (load_config, 
                build_dataset,
                add_quality_flags,
                graph_tables,
                make_qa_report,
                load_iarc_agents,
                merge_seed_with_iarc_expansion,
                DEFAULT_IARC_URL,
                read_seed_csv,
                write_csv,
                write_json,
                write_jsonl,
                write_dataframe_json,
                )



In [2]:
# ===== Editable Parameters =====
seed_csv = "data/seed.csv"  # curated starting set; replace with your seed file
out_dir = "outputs"
cache_dir = "cache"
config_path = "config.yaml"

# Enrichment toggles
use_pubchem = True
use_classyfire = True   # ClassyFire / ChemOnt validated chemical class
expand_iarc = True

# IARC expansion options
iarc_source = DEFAULT_IARC_URL  # or local file path like "data/iarc_export.xlsx"
include_group3 = False
iarc_groups = ["Group 1", "Group 2A", "Group 2B", "Group 3"]

# Optional local sources (set to None to skip)
iarc_csv = None
ntp_csv = None
oehha_csv = None
ctd_diseases = None
ctd_genes = None

# Output format: "csv", "json", or "both"
output_format = "both"

In [3]:
# ===== Pipeline Run =====
out_path = Path(out_dir)
cache_path = Path(cache_dir)
out_path.mkdir(parents=True, exist_ok=True)
cache_path.mkdir(parents=True, exist_ok=True)

cfg = load_config(config_path)
seed = read_seed_csv(seed_csv)
original_seed_count = len(seed)
iarc_skipped = None

if expand_iarc:
    iarc_agents = load_iarc_agents(
        iarc_source,
        include_groups=iarc_groups,
        include_group3=include_group3,
    )
    seed, iarc_skipped = merge_seed_with_iarc_expansion(seed, iarc_agents)
    print(f"Loaded {len(iarc_agents)} IARC records and appended {len(seed) - original_seed_count} new rows")

df, provenance = build_dataset(
    seed,
    cfg=cfg,
    cache_dir=cache_path / "pubchem",
    iarc_csv=iarc_csv,
    ntp_csv=ntp_csv,
    oehha_csv=oehha_csv,
    ctd_diseases=ctd_diseases,
    ctd_genes=ctd_genes,
    use_pubchem=use_pubchem,
    use_classyfire=use_classyfire,
    classyfire_cache_dir=cache_path / "classyfire",
)

df = add_quality_flags(df, cfg)
report = make_qa_report(df)
graph = graph_tables(df)

write_csv(df, out_path / "carcinogens_enriched.csv")
if output_format in {"json", "both"}:
    write_dataframe_json(df, out_path / "carcinogens_enriched.json")

if "needs_manual_review" in df.columns:
    write_csv(df.loc[df["needs_manual_review"]].copy(), out_path / "manual_review_queue.csv")

if iarc_skipped is not None:
    write_csv(iarc_skipped, out_path / "iarc_expansion_skipped_duplicates.csv")

write_jsonl(provenance, out_path / "source_provenance.jsonl")
write_json(report, out_path / "qa_report.json")

for name, table in graph.items():
    write_csv(table, out_path / f"{name}.csv")
    if output_format in {"json", "both"}:
        write_dataframe_json(table, out_path / f"{name}.json")

print(f"Wrote {len(df)} rows to {out_path / 'carcinogens_enriched.csv'}")
print(f"Manual review count: {report['manual_review_count']}")

Loaded 1058 IARC records and appended 1048 new rows
Wrote 1050 rows to outputs/carcinogens_enriched.csv
Manual review count: 209


In [4]:
# ===== Quick Inspection =====
qa = json.loads((Path(out_dir) / "qa_report.json").read_text(encoding="utf-8"))
print("Rows:", qa.get("row_count"))
print("Manual review:", qa.get("manual_review_count"))

preview_cols = [
    c for c in [
        "standard_node_id",
        "srandard_node_id",
        "carcinogen_id",
        "display_label",
        "iarc_group_normalized",
        "norm_carcinogen_class",
        "carcinogen_class_normalized",
        "carcinogen_class_source",
        "chemont_class",
        "chemont_superclass",
        "classyfire_validated",
        "pubchem_cid",
        "casrn",
        "needs_manual_review",
    ]
    if c in df.columns
]

df[preview_cols].head(20)

Rows: 1050
Manual review: 209


,standard_node_id,srandard_node_id,carcinogen_id,display_label,iarc_group_normalized,norm_carcinogen_class,carcinogen_class_normalized,carcinogen_class_source,chemont_class,chemont_superclass,classyfire_validated,pubchem_cid,casrn,needs_manual_review
0,4ABP,4ABP,4ABP,4-Aminobiphenyl,Group 1,Aromatic_Amine,Aromatic_Amine,seed_csv,Benzene and substituted derivatives,Benzenoids,True,7102,92-67-1,False
1,AFB1,AFB1,AFB1,Aflatoxin B1,Group 1,Mycotoxin,Mycotoxin,seed_csv,Coumarins and derivatives,Phenylpropanoids and polyketides,False,186907,1162-65-8,False
2,IARC_1_1_1_2_Tetrachloroethane,IARC_1_1_1_2_Tetrachloroethane,IARC_1_1_1_2_Tetrachloroethane,"1,1,1,2-Tetrachloroethane",Group 2B,Organochlorine,Organochlorine,classyfire,Organochlorides,Organohalogen compounds,True,12418,630-20-6,False
3,IARC_1_1_1_Trichloroethane,IARC_1_1_1_Trichloroethane,IARC_1_1_1_Trichloroethane,"1,1,1-Trichloroethane",Group 2A,Organochlorine,Organochlorine,classyfire,Organochlorides,Organohalogen compounds,True,6278,71-55-6,False
4,IARC_1_1_2_2_Tetrachloroethane,IARC_1_1_2_2_Tetrachloroethane,IARC_1_1_2_2_Tetrachloroethane,"1,1,2,2-Tetrachloroethane",Group 2B,Organochlorine,Organochlorine,classyfire,Organochlorides,Organohalogen compounds,True,6591,79-34-5,False
5,IARC_1_1_2_Trichloroethane,IARC_1_1_2_Trichloroethane,IARC_1_1_2_Trichloroethane,"1,1,2-Trichloroethane",Group 3,Organochlorine,Organochlorine,classyfire,Organochlorides,Organohalogen compounds,True,6574,79-00-5,False
6,IARC_1_1_Dimethylhydrazine,IARC_1_1_Dimethylhydrazine,IARC_1_1_Dimethylhydrazine,"1,1-Dimethylhydrazine",Group 2B,Alkylhydrazines,Alkylhydrazines,chemont_class,Organonitrogen compounds,Organic nitrogen compounds,True,5976,57-14-7,False
7,IARC_1_2_3_Trichloropropane,IARC_1_2_3_Trichloropropane,IARC_1_2_3_Trichloropropane,"1,2,3-Trichloropropane",Group 2A,Organochlorine,Organochlorine,classyfire,Organochlorides,Organohalogen compounds,True,7285,96-18-4,False
8,IARC_1_2_3_Tris_chloromethoxy_propane,IARC_1_2_3_Tris_chloromethoxy_propane,IARC_1_2_3_Tris_chloromethoxy_propane,"1,2,3-Tris(chloromethoxy)propane",Group 3,Organochlorine,Organochlorine,classyfire,Glycerolipids,Lipids and lipid-like molecules,True,62896,38571-73-2,False
9,IARC_1_2_Bis_chloromethoxy_ethane,IARC_1_2_Bis_chloromethoxy_ethane,IARC_1_2_Bis_chloromethoxy_ethane,"1,2-Bis(chloromethoxy)ethane",Group 3,Organochlorine,Organochlorine,classyfire,Organooxygen compounds,Organic oxygen compounds,True,61632,13483-18-6,False
